# **CODE 8c v7 COMPLETE: COMPREHENSIVE BACKTESTING (UPDATED)**

---

## **🔄 UPDATES IN v7:**

### **1. ATR Multiplier: 4.0× (changed from 1.5×)**
### **2. Transaction Cost: 0.25% (both buy and sell)**
### **3. Stop-Loss Logic:**
- Convert 4×ATR to percentage of entry price
- Apply percentage to peak price (trailing stop)
- Check if **intraday Low** touches trigger
- Exit at **trigger price** (not close)

### **4. Return Calculation:**
- Entry price adjusted: +0.25%
- Exit price adjusted: -0.25%
- Returns calculated using adjusted prices

---

## **📦 COMPLETE OUTPUTS (10 FILES):**

### **1 Comprehensive Excel File:**
- **backtest_comprehensive_report.xlsx** (11 sheets)
  - Summary
  - Year-wise Returns
  - Cumulative Returns
  - Detailed Metrics
  - **ALL Trades** (complete transaction log)
  - **ALL Cashflows** (dated entries/exits)
  - Stock Performance
  - Individual label sheets

### **9 CSV Files:**
- trades_High.csv, trades_Medium.csv, trades_Low.csv, trades_Ignore.csv
- cashflows_High.csv, cashflows_Medium.csv, cashflows_Low.csv, cashflows_Ignore.csv
- stock_performance.csv

**All files auto-download to Windows Downloads folder**

---

## **⏱️ RUNTIME:** 45-60 minutes

---

## **STEP 0: Mount Google Drive**

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

print("\n" + "="*80)
print("✓ Google Drive mounted")
print("="*80)

Mounted at /content/drive

✓ Google Drive mounted


## **STEP 1: Install Packages**

In [2]:
!pip install xlsxwriter -q

print("✓ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.4 MB/s eta 0:00:00
✓ Packages installed


## **STEP 2: Import Libraries**

In [3]:
import pandas as pd
import numpy as np
import pickle
import warnings
import time
from datetime import datetime, timedelta
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import xlsxwriter
from scipy.optimize import newton

warnings.filterwarnings('ignore')

print("="*80)
print("CODE 8c v7 COMPLETE: COMPREHENSIVE BACKTESTING (UPDATED)")
print("="*80)
print()
print("✓ Libraries imported")
print(f"  Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("🔄 UPDATES IN v7:")
print("  • ATR Multiplier: 4.0× (changed from 1.5×)")
print("  • Transaction Cost: 0.25% (both buy and sell)")
print("  • Stop-loss: 4×ATR as % of entry, applied to peak")
print("  • Exit trigger: When intraday Low touches trigger")
print("  • Exit price: Trigger price (not close)")
print()
print("📦 OUTPUTS:")
print("  • 1 comprehensive Excel (11 sheets)")
print("  • 9 detailed CSV files")
print("  • All files auto-download")
print()

CODE 8c v7 COMPLETE: COMPREHENSIVE BACKTESTING (UPDATED)

✓ Libraries imported
  Time: 2026-08-18 06:14:55

🔄 UPDATES IN v7:
  • ATR Multiplier: 4.0× (changed from 1.5×)
  • Transaction Cost: 0.25% (both buy and sell)
  • Stop-loss: 4×ATR as % of entry, applied to peak
  • Exit trigger: When intraday Low touches trigger
  • Exit price: Trigger price (not close)

📦 OUTPUTS:
  • 1 comprehensive Excel (11 sheets)
  • 9 detailed CSV files
  • All files auto-download



## **STEP 3: Configuration**

**⚠️ UPDATE THESE PATHS**

In [4]:
# ============================================================================
# UPDATE THESE PATHS
# ============================================================================

DRIVE_FOLDER = '/content/drive/MyDrive/masters/'  # ← UPDATE THIS
MODEL_FOLDER = '/content/'                         # ← Models uploaded to Colab

# Stock universe (must match training)
SUFFIX = 'midcap150'  # ← UPDATE: 'all', 'nifty100', 'midcap150', 'smallcap'

# File paths (note: no single best_model.pkl — all models come from Drive via
# COMBO_MODELS_DIR in the next cell). Shared fileset is also taken from there.
TEST1_FILE = os.path.join(DRIVE_FOLDER, f'test1_data_{SUFFIX}.parquet')
TEST2_FILE = os.path.join(DRIVE_FOLDER, f'test2_data_{SUFFIX}.parquet')
METADATA_FILE = os.path.join(DRIVE_FOLDER, f'feature_metadata_{SUFFIX}.csv')
# FEATURE_COLS_FILE & ENCODERS_FILE are set from COMBO_MODELS_DIR in the next cell.

# ============================================================================
# UPDATED TRADING PARAMETERS
# ============================================================================

INVESTMENT_PER_TRADE = 100000  # ₹1 lakh
ATR_MULTIPLIER = 4.0           # 4× ATR (UPDATED from 1.5)
TRANSACTION_COST = 0.25        # 0.25% transaction cost (NEW)

print("-" * 80)
print("CONFIGURATION")
print("-" * 80)
print(f"Stock universe: {SUFFIX}")
print(f"Investment/trade: ₹{INVESTMENT_PER_TRADE:,}")
print(f"\n🔄 UPDATED PARAMETERS:")
print(f"  ATR multiplier: {ATR_MULTIPLIER}x (changed from 1.5x)")
print(f"  Transaction cost: {TRANSACTION_COST}% (NEW)")
print(f"    - Entry: +{TRANSACTION_COST}% (pay more to buy)")
print(f"    - Exit: -{TRANSACTION_COST}% (receive less on sell)")
print()

# Verify files
print("Checking files...")
for filepath, desc in [
    (TEST1_FILE, "Test1 data"),
    (TEST2_FILE, "Test2 data"),
    (METADATA_FILE, "Metadata"),
]:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{desc} not found: {filepath}")
    print(f"  ✓ {desc}")

print("\n✓ Configuration complete")
print()

--------------------------------------------------------------------------------
CONFIGURATION
--------------------------------------------------------------------------------
Stock universe: midcap150
Investment/trade: ₹100,000

🔄 UPDATED PARAMETERS:
  ATR multiplier: 4.0x (changed from 1.5x)
  Transaction cost: 0.25% (NEW)
    - Entry: +0.25% (pay more to buy)
    - Exit: -0.25% (receive less on sell)

Checking files...
  ✓ Test1 data
  ✓ Test2 data
  ✓ Metadata

✓ Configuration complete



## **STEP 3b: Batch Configuration**

Point `COMBO_MODELS_DIR` at the `combo_models/` folder from Code 8b's zip (upload it to your Google Drive masters folder). The notebook runs the FULL backtest for EVERY `model_combo_*.pkl`,
writing each model's complete output (Excel + 9 CSVs) into its own subfolder, then
compiles one cross-model comparison workbook of per-segment, per-predicted-class IRRs.

In [5]:
# ── Batch settings ──────────────────────────────────────────────────────────
# All models + shared fileset come from Google Drive. Upload the combo_models/
# folder (from Code 8b's zip) into your Drive masters folder.
COMBO_MODELS_DIR = os.path.join(DRIVE_FOLDER, 'combo_models/')  # ← in Google Drive
BATCH_OUTPUT_DIR = '/content/batch_outputs/'   # per-model subfolders go here
os.makedirs(BATCH_OUTPUT_DIR, exist_ok=True)

# The shared fileset (encoders, label_map, feature_cols) is the SAME for all combos.
# 8b saves a copy inside combo_models/, AND the same files exist in the main Drive
# folder. Resolve each from combo_models/ first, then fall back to DRIVE_FOLDER —
# so the batch works whether or not the shared files were uploaded into combo_models/.
def _resolve_shared(fname):
    for base in [COMBO_MODELS_DIR, DRIVE_FOLDER]:
        cand = os.path.join(base, fname)
        if os.path.exists(cand):
            return cand
    return None

ENCODERS_FILE     = _resolve_shared('categorical_encoders.pkl')
FEATURE_COLS_FILE = _resolve_shared('feature_columns_model.pkl')
LABELMAP_FILE     = _resolve_shared('label_mapping.pkl')
UTILITY_FILE      = _resolve_shared('utility_matrix.pkl')   # for the EU decision rule

import glob
combo_model_files = sorted(glob.glob(os.path.join(COMBO_MODELS_DIR, 'model_combo_*.pkl')))
if not combo_model_files:
    raise FileNotFoundError(f"No model_combo_*.pkl in {COMBO_MODELS_DIR}. "
                            f"Upload the combo_models/ folder from Code 8b to your Drive.")

print(f"Found {len(combo_model_files)} combo models:")
for p in combo_model_files:
    print(f"  • {os.path.basename(p)}")

print(f"\nShared fileset resolution (combo_models/ first, then {DRIVE_FOLDER}):")
_missing = []
for nm, fp in [('categorical_encoders.pkl', ENCODERS_FILE),
               ('feature_columns_model.pkl', FEATURE_COLS_FILE),
               ('label_mapping.pkl', LABELMAP_FILE),
               ('utility_matrix.pkl', UTILITY_FILE)]:
    if fp:
        print(f"  ✓ {nm:32s} → {fp}")
    else:
        print(f"  ✗ {nm:32s} → NOT FOUND in combo_models/ or {DRIVE_FOLDER}")
        _missing.append(nm)
_missing = [m for m in _missing if m != 'utility_matrix.pkl']  # optional: locked fallback exists
if _missing:
    raise FileNotFoundError(
        f"Missing shared file(s): {_missing}. These are saved by Code 8b — copy them "
        f"into either {COMBO_MODELS_DIR} or {DRIVE_FOLDER} and re-run. "
        f"(They are identical for all combo models.)")

Found 6 combo models:
  • model_combo_062.pkl
  • model_combo_068.pkl
  • model_combo_071.pkl
  • model_combo_074.pkl
  • model_combo_077.pkl
  • model_combo_080.pkl

Shared fileset resolution (combo_models/ first, then /content/drive/MyDrive/masters/):
  ✓ categorical_encoders.pkl         → /content/drive/MyDrive/masters/combo_models/categorical_encoders.pkl
  ✓ feature_columns_model.pkl        → /content/drive/MyDrive/masters/combo_models/feature_columns_model.pkl
  ✓ label_mapping.pkl                → /content/drive/MyDrive/masters/combo_models/label_mapping.pkl
  ✓ utility_matrix.pkl               → /content/drive/MyDrive/masters/combo_models/utility_matrix.pkl


## **STEP 3c: Define the single-model backtest function**

This wraps the existing backtest steps (load → predict → simulate → CSVs → summaries →
Excel) **verbatim** — only the model source and output paths are parameterized, so the
backtest logic is identical to the single-model notebook.

In [6]:
def run_one_backtest(MODEL_FILE, OUT_DIR, MODEL_LABEL):
    """FULL existing backtest for one model into OUT_DIR. Logic identical to the
    single-model notebook; only model source + output paths parameterized.
    Returns comparison rows (per segment, per predicted-class IRR + count)."""
    import os
    os.makedirs(OUT_DIR, exist_ok=True)
    print("\n" + "#"*80); print(f"# BACKTEST: {MODEL_LABEL}"); print("#"*80)
    # ===== [cell 10] =====
    print("-" * 80)
    print("LOADING MODEL")
    print("-" * 80)

    with open(MODEL_FILE, 'rb') as f:
        model = pickle.load(f)
    print(f"✓ Model: {type(model).__name__}")

    with open(ENCODERS_FILE, 'rb') as f:
        label_encoders = pickle.load(f)
    print(f"✓ Encoders: {len(label_encoders)} features")

    # Load label_mapping.pkl saved by Code 8b (fixes encode/decode mismatch)
    label_mapping_from_8b = None
    label_mapping_file = LABELMAP_FILE   # resolved: combo_models/ first, then masters/
    if os.path.exists(label_mapping_file):
        with open(label_mapping_file, 'rb') as f:
            label_mapping_from_8b = pickle.load(f)
        print(f"✓ label_mapping.pkl loaded from Code 8b")
        print(f"  Mapping: {label_mapping_from_8b}")
    else:
        print("⚠️ label_mapping.pkl not found in combo_models/ or masters/ — falling back to alphabetical")
        print("   (Upload label_mapping.pkl from Code 8b output to fix this)")

    # Keep target_encoder for backward compatibility
    if 'conviction_label' in label_encoders:
        target_encoder = label_encoders['conviction_label']
    else:
        target_encoder = None

    print()

    # ===== [cell 12] =====
    print("-" * 80)
    print("LOADING TEST DATA")
    print("-" * 80)

    # Load metadata
    metadata_df = pd.read_csv(METADATA_FILE)
    feature_columns = metadata_df['feature_name'].tolist()
    print(f"Features: {len(feature_columns)}")

    # Load Test1
    print("\nLoading Test1 (2020-2022)...")
    test1_df = pd.read_parquet(TEST1_FILE)
    test1_df['test_period'] = 'Test1'
    test1_df['period_end'] = datetime(2022, 12, 31)
    print(f"  ✓ {len(test1_df):,} rows")

    # Load Test2
    print("\nLoading Test2 (2023-2025)...")
    test2_df = pd.read_parquet(TEST2_FILE)
    test2_df['test_period'] = 'Test2'
    test2_df['period_end'] = datetime(2025, 12, 31)
    print(f"  ✓ {len(test2_df):,} rows")

    # Combine
    test_df = pd.concat([test1_df, test2_df], ignore_index=True)
    print(f"\nTotal: {len(test_df):,} rows")

    # Standardize column names
    date_col = None
    for col in ['Date', 'date', 'DATE', 'trading_date']:
        if col in test_df.columns:
            date_col = col
            if col != 'Date':
                test_df['Date'] = test_df[col]
            break

    symbol_col = None
    for col in ['Ticker', 'ticker', 'Symbol', 'symbol']:
        if col in test_df.columns:
            symbol_col = col
            if col != 'Ticker':
                test_df['Ticker'] = test_df[col]
            break

    # Convert dates
    test_df['Date'] = pd.to_datetime(test_df['Date'])

    # Standardize ATR column name — Code 3/6 name it 'ATR_14'
    if 'ATR' not in test_df.columns:
        for atr_alt in ['ATR_14', 'atr_14', 'ATR14', 'atr']:
            if atr_alt in test_df.columns:
                test_df['ATR'] = test_df[atr_alt]
                print(f"  ✓ Using '{atr_alt}' as ATR for exit calculation")
                break

    # Check required columns
    required = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'ATR', 'conviction_label']
    missing = [col for col in required if col not in test_df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    print(f"\nDate range: {test_df['Date'].min()} to {test_df['Date'].max()}")
    print(f"Unique stocks: {test_df['Ticker'].nunique()}")
    print()

    # Show actual label distribution
    print("Actual label distribution:")
    for label in sorted(test_df['conviction_label'].unique()):
        count = (test_df['conviction_label'] == label).sum()
        pct = count / len(test_df) * 100
        print(f"  {label:10s}: {count:>8,} ({pct:5.2f}%)")
    print()

    # ===== [cell 14] =====
    print("-" * 80)
    print("MAKING PREDICTIONS")
    print("-" * 80)

    # Use the EXACT feature list the model was trained on (saved by Code 8b).
    # This guarantees identical features/order and respects xgb_relevant drops,
    # the DROP_INTER_STOCK toggle, and any pruning applied during training.
    with open(FEATURE_COLS_FILE, 'rb') as f:
        model_feature_columns = pickle.load(f)
    print(f"  Loaded {len(model_feature_columns)} model features from feature_columns_model.pkl")

    missing_feats = [c for c in model_feature_columns if c not in test_df.columns]
    if missing_feats:
        raise ValueError(f"Test data missing {len(missing_feats)} model features: {missing_feats[:10]}")

    X_test = test_df[model_feature_columns].copy()
    y_test = test_df['conviction_label'].copy()

    # Encode categorical
    print("\nEncoding features...")
    for col in X_test.columns:
        if X_test[col].dtype == 'object' or X_test[col].dtype.name == 'category':
            if col in label_encoders:
                le = label_encoders[col]
                known = set(le.classes_)
                X_test[col] = X_test[col].apply(
                    lambda x: x if x in known else le.classes_[0]
                )
                X_test[col] = le.transform(X_test[col])
            else:
                X_test[col] = pd.factorize(X_test[col])[0]

    # Convert to numeric; replace inf→NaN (XGBoost rejects inf) but KEEP NaN
    # (model was trained with NaN in keep_nan features; XGBoost handles them natively)
    X_test = X_test.apply(pd.to_numeric, errors='coerce')
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    print(f"  inf→NaN done; NaN retained for native XGBoost handling "
          f"({int(X_test.isnull().sum().sum()):,} NaN cells kept)")

    # Predict
    print("Generating predictions...")
    start = time.time()

    y_pred_proba = model.predict_proba(X_test)

    # ── EXPECTED-UTILITY DECISION RULE (replaces argmax) ────────────────────
    # argmax picks the most LIKELY class; the EU rule picks the most VALUABLE
    # one, given the cost/benefit of each (actual, predicted) pair. This must
    # match the rule Code 8b used to select hyperparameters.
    UTILITY_MATRIX_FALLBACK = np.array([   # locked matrix (rows=actual, cols=pred)
        [    2,    -15,   -30,   -46],     # Actual Ignore
        [    0,     10,   -11,   -22],     # Actual Low
        [   -5,      5,    30,   -11],     # Actual Medium
        [  -15,      6,    16,    50],     # Actual High
    ], dtype=float)
    if UTILITY_FILE and os.path.exists(UTILITY_FILE):
        with open(UTILITY_FILE, 'rb') as _f:
            U_DECIDE = np.asarray(pickle.load(_f), dtype=float)
        print(f"✓ utility_matrix.pkl loaded from Code 8b: {UTILITY_FILE}")
    else:
        U_DECIDE = UTILITY_MATRIX_FALLBACK
        print("⚠️ utility_matrix.pkl not found — using locked fallback matrix")

    y_pred = (y_pred_proba @ U_DECIDE).argmax(axis=1)
    print(f"  Decision rule: expected utility (not argmax)")

    print(f"✓ Completed in {time.time()-start:.1f}s")

    # Decode predictions
    print("\nDecoding predictions to label names...")
    print(f"  Predicted type: {type(y_pred[0])}")
    print(f"  Actual label type: {type(y_test.iloc[0])}")

    if isinstance(y_pred[0], (int, np.integer)):
        print("  Predictions are encoded integers - decoding...")

        if label_mapping_from_8b is not None:
            # Use the exact mapping saved by Code 8b — correct fix
            reverse_map    = {v: k for k, v in label_mapping_from_8b.items()}
            y_pred_decoded = [reverse_map.get(int(p), str(p)) for p in y_pred]
            print(f"  ✓ Using label_mapping.pkl from Code 8b (correct)")
            print(f"  Mapping: {reverse_map}")
        elif target_encoder is not None:
            y_pred_decoded = target_encoder.inverse_transform(y_pred)
            print(f"  ✓ Using saved label encoder (fallback)")
            print(f"  Mapping: {dict(enumerate(target_encoder.classes_))}")
        else:
            # Last resort alphabetical fallback — warn the user clearly
            label_mapping = {0: 'Ignore', 1: 'Low', 2: 'Medium', 3: 'High'}  # Code 7 order
            y_pred_decoded = [label_mapping.get(pred, str(pred)) for pred in y_pred]
            print(f"  ⚠️ Using alphabetical fallback — upload label_mapping.pkl to fix")
            print(f"  Mapping: {label_mapping}")
    else:
        y_pred_decoded = y_pred
        print("  ✓ Predictions already in label format")

    # Add to dataframe
    test_df['predicted_label'] = y_pred_decoded
    test_df['prediction_confidence'] = y_pred_proba.max(axis=1)

    # Show distribution
    print("\nPredicted distribution:")
    for label in sorted(pd.Series(y_pred_decoded).unique()):
        count = (pd.Series(y_pred_decoded) == label).sum()
        pct = count / len(y_pred_decoded) * 100
        print(f"  {label:10s}: {count:>8,} ({pct:5.2f}%)")

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred_decoded)
    print(f"\nOverall accuracy: {accuracy*100:.2f}%")

    # Confusion matrix
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_decoded, labels=['High', 'Medium', 'Low', 'Ignore'])
    cm_df = pd.DataFrame(cm,
                         index=['True_' + l for l in ['High', 'Medium', 'Low', 'Ignore']],
                         columns=['Pred_' + l for l in ['High', 'Medium', 'Low', 'Ignore']])
    print(cm_df)

    print()

    # ===== [cell 16] =====
    def calculate_exit(stock_df, entry_date, entry_price, atr, period_end):
        """
        Calculate exit using 4× ATR percentage-based trailing stop from peak.

        UPDATED LOGIC:
        - Convert 4×ATR to percentage of entry price
        - Track peak price from entry close
        - Exit when intraday Low touches trigger
        - Exit at trigger price (not close)
        """
        # Calculate stop-loss percentage
        stop_loss_threshold = ATR_MULTIPLIER * atr  # 4 × ATR
        stop_loss_pct = (stop_loss_threshold / entry_price) * 100

        # Get future data
        future = stock_df[stock_df['Date'] > entry_date].sort_values('Date')
        future = future[future['Date'] <= period_end]

        if len(future) == 0:
            # No future data - force exit at period end
            period_data = stock_df[stock_df['Date'] <= period_end]
            if len(period_data) > 0:
                last_row = period_data.iloc[-1]
                return last_row['Date'], last_row['Close'], 'Forced_Period_End', stop_loss_pct
            else:
                return entry_date, entry_price, 'No_Data', stop_loss_pct

        # Track peak starting from entry close
        running_max_price = entry_price

        for _, row in future.iterrows():
            # Update peak with current high (could go higher intraday)
            running_max_price = max(running_max_price, row['High'])

            # Calculate trigger price based on peak
            trigger_price = running_max_price * (1 - stop_loss_pct / 100)

            # Check if intraday Low touched the trigger
            if row['Low'] <= trigger_price:
                # Exit at trigger price (more realistic than close)
                return row['Date'], trigger_price, 'ATR', stop_loss_pct

        # Didn't hit exit - force at period end
        last_row = future.iloc[-1]
        return last_row['Date'], last_row['Close'], 'Forced_Period_End', stop_loss_pct

    print("✓ Exit calculation function defined (UPDATED)")
    print()
    print("Exit Logic:")
    print(f"  • Stop-loss: {ATR_MULTIPLIER}× ATR converted to %")
    print(f"  • Track peak from entry close")
    print(f"  • Trigger = Peak × (1 - stop_loss_%)")
    print(f"  • Exit when: Intraday Low <= Trigger")
    print(f"  • Exit price: Trigger price")
    print(f"  • Transaction cost applied separately to returns")
    print()

    # ===== [cell 18] =====
    def xirr(cash_flows):
        """
        Calculate XIRR for dated cash flows.
        """
        if len(cash_flows) < 2:
            return 0.0

        cash_flows = sorted(cash_flows, key=lambda x: x[0])
        start_date = cash_flows[0][0]

        years = []
        amounts = []

        for date, amount in cash_flows:
            days_diff = (date - start_date).days
            year_diff = days_diff / 365.25
            years.append(year_diff)
            amounts.append(amount)

        if all(x > 0 for x in amounts) or all(x < 0 for x in amounts):
            return 0.0

        def npv(rate):
            return sum(amt / (1 + rate) ** yr for yr, amt in zip(years, amounts))

        try:
            rate = newton(npv, 0.1, maxiter=100)
            return rate * 100
        except:
            for guess in [0.01, 0.05, 0.2, 0.3, -0.1, -0.2]:
                try:
                    rate = newton(npv, guess, maxiter=100)
                    return rate * 100
                except:
                    continue

        return 0.0

    print("✓ XIRR function defined")
    print()

    # ===== [cell 20] =====
    print("="*80)
    print("TRADING SIMULATION WITH UPDATED LOGIC")
    print("="*80)
    print()
    print("Calculating exits with new parameters...")
    print(f"  • 4× ATR stop-loss (percentage-based)")
    print(f"  • 0.25% transaction cost")
    print(f"  • Exit at trigger price when Low touches it")
    print()
    print("This will take 30-40 minutes...")
    print()

    LABELS = ['High', 'Medium', 'Low', 'Ignore']
    all_results = {}
    all_trades = {}
    all_cashflows = {}
    stock_performance = []

    # Transaction cost as decimal
    tc_decimal = TRANSACTION_COST / 100  # 0.0025

    for label_idx, label in enumerate(LABELS):
        print(f"\n{'='*80}")
        print(f"SIMULATING: {label} ({label_idx+1}/4)")
        print(f"{'='*80}")

        start_time = time.time()

        # Get predictions
        label_pred = test_df[test_df['predicted_label'] == label].copy()
        print(f"\nPredictions: {len(label_pred):,}")

        if len(label_pred) == 0:
            print(f"⚠️ No predictions, skipping...")
            continue

        # Calculate exits
        trades = []
        cashflows = []

        print("Calculating exits...")
        calc_start = time.time()

        trade_counter = 0

        for idx, row in label_pred.iterrows():
            # Progress
            if (len(trades) + 1) % 1000 == 0:
                elapsed = time.time() - calc_start
                rate = len(trades) / elapsed
                remaining = (len(label_pred) - len(trades)) / rate
                print(f"  {len(trades):,}/{len(label_pred):,} ({len(trades)/len(label_pred)*100:.1f}%) - "
                      f"ETA: {remaining/60:.1f} min")

            trade_counter += 1
            trade_id = f"{label}_{trade_counter:06d}"

            signal_date = row['Date']            # features/prediction date (= t-1 after Code 6 shift)
            symbol = row['Ticker']
            atr = row['ATR']
            test_period = row['test_period']
            period_end = row['period_end']
            actual_label = row['conviction_label']

            # Get this stock's data
            stock_data = test_df[test_df['Ticker'] == symbol].sort_values('Date').copy()

            # NEXT-BAR ENTRY: Code 6 shifted labels so the label on the signal-date row
            # describes the trade ENTERED ON THE NEXT TRADING BAR. So we buy at the next
            # bar's Close, not the signal date's Close. This makes the simulated return
            # match the return Code 6 used to assign the conviction label (no look-ahead).
            next_bars = stock_data[stock_data['Date'] > signal_date]
            if len(next_bars) == 0:
                # No next bar (last date for this ticker) — cannot trade; skip, matching
                # Code 6 dropping the last row per ticker after the shift.
                continue
            entry_row = next_bars.iloc[0]
            entry_date = entry_row['Date']
            entry_price_raw = entry_row['Close']

            # Calculate exit from the entry bar onward
            exit_date, exit_price_raw, exit_type, stop_loss_pct = calculate_exit(
                stock_data, entry_date, entry_price_raw, atr, period_end
            )

            # Apply transaction costs
            entry_price_adjusted = entry_price_raw * (1 + tc_decimal)  # Pay MORE to buy
            exit_price_adjusted = exit_price_raw * (1 - tc_decimal)    # Receive LESS on sell

            # Trade metrics using ADJUSTED prices
            shares = INVESTMENT_PER_TRADE / entry_price_adjusted
            proceeds = shares * exit_price_adjusted
            profit = proceeds - INVESTMENT_PER_TRADE
            return_pct = ((exit_price_adjusted / entry_price_adjusted) - 1) * 100
            days_held = (exit_date - entry_date).days

            if days_held == 0:
                days_held = 1

            # Annualized return
            annualized_return = ((exit_price_adjusted / entry_price_adjusted) ** (365.25 / days_held) - 1) * 100

            # Store trade
            trades.append({
                'trade_id': trade_id,
                'entry_date': entry_date,
                'exit_date': exit_date,
                'symbol': symbol,
                'predicted_label': label,
                'actual_label': actual_label,
                'entry_price_raw': entry_price_raw,
                'entry_price_adjusted': entry_price_adjusted,
                'exit_price_raw': exit_price_raw,
                'exit_price_adjusted': exit_price_adjusted,
                'transaction_cost_pct': TRANSACTION_COST,
                'shares': shares,
                'investment': INVESTMENT_PER_TRADE,
                'proceeds': proceeds,
                'profit': profit,
                'return_pct': return_pct,
                'annualized_return': annualized_return,
                'days_held': days_held,
                'test_period': test_period,
                'exit_type': exit_type,
                'atr': atr,
                'stop_loss_pct': stop_loss_pct,
                'atr_threshold': ATR_MULTIPLIER * atr,
                'entry_year': entry_date.year,
                'exit_year': exit_date.year
            })

            # Store cashflows
            cashflows.append({
                'trade_id': trade_id,
                'date': entry_date,
                'symbol': symbol,
                'flow_type': 'Investment',
                'amount': -INVESTMENT_PER_TRADE,
                'shares': shares,
                'price': entry_price_adjusted,
                'description': f'Buy {shares:.2f} shares @ ₹{entry_price_adjusted:.2f} (incl 0.25% cost)'
            })

            cashflows.append({
                'trade_id': trade_id,
                'date': exit_date,
                'symbol': symbol,
                'flow_type': f'Exit_{exit_type}',
                'amount': proceeds,
                'shares': shares,
                'price': exit_price_adjusted,
                'description': f'Sell {shares:.2f} shares @ ₹{exit_price_adjusted:.2f} ({exit_type}, after 0.25% cost)'
            })

        trades_df = pd.DataFrame(trades)
        cashflows_df = pd.DataFrame(cashflows)

        all_trades[label] = trades_df
        all_cashflows[label] = cashflows_df

        # Calculate stock-level performance
        print("\nCalculating stock-level performance...")
        stock_stats = trades_df.groupby('symbol').agg({
            'trade_id': 'count',
            'profit': ['sum', 'mean', 'std'],
            'return_pct': ['mean', 'median'],
            'days_held': 'mean',
            'annualized_return': 'mean'
        }).reset_index()

        stock_stats.columns = ['symbol', 'num_trades', 'total_profit', 'avg_profit',
                               'std_profit', 'avg_return_pct', 'median_return_pct',
                               'avg_days_held', 'avg_annualized_return']
        stock_stats['label'] = label
        stock_stats['win_rate'] = trades_df.groupby('symbol')['profit'].apply(
            lambda x: (x > 0).sum() / len(x) * 100
        ).values

        stock_performance.append(stock_stats)

        # Exit breakdown
        print(f"\n✓ Exit calculation complete")
        print(f"\nTotal trades: {len(trades_df):,}")
        print(f"  ATR exits: {(trades_df['exit_type']=='ATR').sum():,} "
              f"({(trades_df['exit_type']=='ATR').sum()/len(trades_df)*100:.1f}%)")
        print(f"  Forced exits: {(trades_df['exit_type']=='Forced_Period_End').sum():,} "
              f"({(trades_df['exit_type']=='Forced_Period_End').sum()/len(trades_df)*100:.1f}%)")

        print(f"\nAverage stop-loss %: {trades_df['stop_loss_pct'].mean():.2f}%")
        print(f"Transaction cost impact: ±{TRANSACTION_COST}%")

        # Metrics
        results = {
            'label': label,
            'total_trades': len(trades_df),
            'total_invested': trades_df['investment'].sum(),
            'total_proceeds': trades_df['proceeds'].sum(),
            'total_profit': trades_df['profit'].sum(),
            'win_trades': (trades_df['profit'] > 0).sum(),
            'loss_trades': (trades_df['profit'] < 0).sum(),
            'win_rate': (trades_df['profit'] > 0).sum() / len(trades_df) * 100,
            'avg_return': trades_df['return_pct'].mean(),
            'median_return': trades_df['return_pct'].median(),
            'avg_annualized_return': trades_df['annualized_return'].mean(),
            'avg_days_held': trades_df['days_held'].mean(),
            'best_trade': trades_df['profit'].max(),
            'worst_trade': trades_df['profit'].min(),
        }

        # Year-wise IRR
        print("\nCalculating year-wise IRR...")
        year_irrs = {}

        years = sorted(trades_df['entry_date'].dt.year.unique())
        for year in years:
            year_entries = trades_df[trades_df['entry_date'].dt.year == year].copy()

            if len(year_entries) == 0:
                continue

            year_end = datetime(year, 12, 31)
            cf = []

            for _, t in year_entries.iterrows():
                cf.append((t['entry_date'], -t['investment']))

                if t['exit_date'].year == year:
                    cf.append((t['exit_date'], t['proceeds']))
                else:
                    year_end_data = test_df[
                        (test_df['Ticker'] == t['symbol']) &
                        (test_df['Date'] <= year_end)
                    ]
                    if len(year_end_data) > 0:
                        year_end_price = year_end_data.iloc[-1]['Close']
                        # Apply transaction cost to year-end exit
                        year_end_price_adj = year_end_price * (1 - tc_decimal)
                        forced_proceeds = t['shares'] * year_end_price_adj
                        cf.append((year_end, forced_proceeds))

            year_irr = xirr(cf) if len(cf) >= 2 else 0.0
            year_irrs[year] = year_irr
            print(f"  {year}: {year_irr:>7.2f}% ({len(year_entries):,} trades)")

        results['year_irrs'] = year_irrs

        # Cumulative IRR — MARK-TO-MARKET at the test-period boundary.
        # For any trade still open past the period's last day, close it at that
        # day's market Close (same convention as the year-end marking above).
        # NOTE: in current data no trade crosses the outer boundary, so this is a
        # safety net that keeps the rule exact if data changes.
        print("\nCalculating cumulative IRR (mark-to-market at period end)...")
        PERIOD_END = {'Test1': datetime(2022, 12, 31), 'Test2': datetime(2025, 12, 31)}
        for period in ['Test1', 'Test2']:
            period_trades = trades_df[trades_df['test_period'] == period]

            if len(period_trades) == 0:
                continue

            p_end = PERIOD_END[period]
            cf = []
            for _, t in period_trades.iterrows():
                cf.append((t['entry_date'], -t['investment']))
                if t['exit_date'] <= p_end:
                    cf.append((t['exit_date'], t['proceeds']))
                else:
                    # still open at period end → mark to Close on/just before p_end
                    pe_data = test_df[(test_df['Ticker'] == t['symbol']) &
                                      (test_df['Date'] <= p_end)]
                    if len(pe_data) > 0:
                        pe_price = pe_data.iloc[-1]['Close'] * (1 - tc_decimal)
                        cf.append((p_end, t['shares'] * pe_price))

            cum_irr = xirr(cf) if len(cf) >= 2 else 0.0
            results[f'{period}_cumulative_irr'] = cum_irr

            years = 3 if period == 'Test1' else 4
            print(f"  {period} ({years} years): {cum_irr:>7.2f}%")

        all_results[label] = results

        elapsed = time.time() - start_time
        print(f"\n✓ {label} completed in {elapsed/60:.1f} min")

    # Combine stock performance
    stock_performance_df = pd.concat(stock_performance, ignore_index=True)

    print("\n" + "="*80)
    print("SIMULATION COMPLETED")
    print("="*80)
    print()

    # ===== [cell 23] =====
    print("="*80)
    print("SAVING DETAILED CSV FILES")
    print("="*80)
    print()

    csv_files = []

    # Save trade logs (one per label)
    print("Saving trade logs...")
    for label in LABELS:
        if label in all_trades:
            filename = os.path.join(OUT_DIR, f'trades_{label}.csv')
            all_trades[label].to_csv(filename, index=False)
            csv_files.append(filename)
            print(f"  ✓ trades_{label}.csv ({len(all_trades[label]):,} trades)")

    # Save cashflow logs (one per label)
    print("\nSaving cashflow logs...")
    for label in LABELS:
        if label in all_cashflows:
            filename = f'/content/cashflows_{label}.csv'
            all_cashflows[label].to_csv(filename, index=False)
            csv_files.append(filename)
            print(f"  ✓ cashflows_{label}.csv ({len(all_cashflows[label]):,} entries)")

    # Save stock performance
    print("\nSaving stock performance...")
    stock_perf_file = os.path.join(OUT_DIR, 'stock_performance.csv')
    stock_performance_df.to_csv(stock_perf_file, index=False)
    csv_files.append(stock_perf_file)
    print(f"  ✓ stock_performance.csv ({len(stock_performance_df):,} stocks)")

    print(f"\n✓ Total: {len(csv_files)} CSV files saved")
    print()

    # ===== [cell 25] =====
    print("-" * 80)
    print("CREATING SUMMARY TABLES + CAPITAL TRACKING")
    print("-" * 80)
    print()

    # ========================================================================
    # EXISTING SUMMARY TABLES (UNCHANGED)
    # ========================================================================

    # Executive Summary
    summary_data = []
    for label in LABELS:
        if label not in all_results:
            continue
        r = all_results[label]
        summary_data.append({
            'Label': label,
            'Trades': r['total_trades'],
            'Win Rate %': r['win_rate'],
            'Avg Return %': r['avg_return'],
            'Avg Annualized %': r['avg_annualized_return'],
            'Total Profit (₹)': r['total_profit'],
            'Test1 IRR %': r.get('Test1_cumulative_irr', 0),
            'Test2 IRR %': r.get('Test2_cumulative_irr', 0),
        })

    summary_df = pd.DataFrame(summary_data)
    print("✓ Executive summary created")

    # Year-wise Returns
    years_data = []
    all_years = set()
    for label in LABELS:
        if label in all_results:
            all_years.update(all_results[label]['year_irrs'].keys())

    for year in sorted(all_years):
        row = {'Year': year}
        for label in LABELS:
            if label in all_results:
                row[f'{label} IRR %'] = all_results[label]['year_irrs'].get(year, 0)
        years_data.append(row)

    yearwise_df = pd.DataFrame(years_data)
    print("✓ Year-wise returns created")

    # Cumulative Returns
    cumulative_data = []
    for period in ['Test1', 'Test2']:
        row = {'Period': period, 'Years': 3 if period == 'Test1' else 4}
        for label in LABELS:
            if label in all_results:
                row[f'{label} IRR %'] = all_results[label].get(f'{period}_cumulative_irr', 0)
        cumulative_data.append(row)

    cumulative_df = pd.DataFrame(cumulative_data)
    print("✓ Cumulative returns created")

    # Detailed Metrics
    metrics_data = []
    for label in LABELS:
        if label not in all_results:
            continue
        r = all_results[label]
        trades = all_trades[label]

        metrics_data.append({
            'Label': label,
            'Total Trades': r['total_trades'],
            'Win Trades': r['win_trades'],
            'Loss Trades': r['loss_trades'],
            'Win Rate %': r['win_rate'],
            'Total Invested (₹Cr)': r['total_invested'] / 10000000,
            'Total Proceeds (₹Cr)': r['total_proceeds'] / 10000000,
            'Total Profit (₹Cr)': r['total_profit'] / 10000000,
            'Avg Return %': r['avg_return'],
            'Median Return %': r['median_return'],
            'Avg Annualized %': r['avg_annualized_return'],
            'Best Trade (₹)': r['best_trade'],
            'Worst Trade (₹)': r['worst_trade'],
            'Avg Days Held': r['avg_days_held'],
            'Median Days': trades['days_held'].median(),
            'Std Dev Return %': trades['return_pct'].std(),
        })

    metrics_df = pd.DataFrame(metrics_data)
    print("✓ Detailed metrics created")

    # ========================================================================
    # NEW: CAPITAL TRACKING ANALYSIS
    # ========================================================================

    print("\n" + "="*80)
    print("CALCULATING CAPITAL DEPLOYMENT METRICS")
    print("="*80)

    # Function to calculate capital metrics from cashflows
    def calculate_capital_metrics(cashflows_df, trades_df):
        """
        Calculate daily capital deployment and cumulative P/L
        """
        # Sort cashflows by date
        cf = cashflows_df.sort_values('date').copy()

        # Create daily timeline
        cf['capital_change'] = cf['amount'].apply(lambda x: -x if x < 0 else 0)  # Track investments
        cf['realized_pl'] = cf['amount'].apply(lambda x: x - INVESTMENT_PER_TRADE if x > 0 else 0)  # Track profits

        # Calculate running totals
        cf['cumulative_invested'] = cf['capital_change'].cumsum()
        cf['cumulative_pl'] = cf['realized_pl'].cumsum()

        # Calculate open positions (investments - exits)
        cf['position_delta'] = cf['flow_type'].apply(lambda x: 1 if x == 'Investment' else -1)
        cf['open_positions'] = cf['position_delta'].cumsum()
        cf['capital_deployed'] = cf['open_positions'] * INVESTMENT_PER_TRADE

        return cf

    # Calculate for each label
    capital_timelines = {}
    capital_summary = []

    for label in LABELS:
        if label not in all_cashflows:
            continue

        print(f"\nProcessing {label}...")

        # Calculate timeline
        timeline = calculate_capital_metrics(all_cashflows[label], all_trades[label])
        capital_timelines[label] = timeline

        # Calculate summary metrics
        summary = {
            'Label': label,
            'Max Capital (₹Cr)': timeline['capital_deployed'].max() / 10000000,
            'Avg Capital (₹Cr)': timeline['capital_deployed'].mean() / 10000000,
            'Peak Positions': int(timeline['open_positions'].max()),
            'Total Invested (₹Cr)': timeline['cumulative_invested'].max() / 10000000,
            'Final P/L (₹Cr)': timeline['cumulative_pl'].iloc[-1] / 10000000 if len(timeline) > 0 else 0,
        }
        capital_summary.append(summary)

        print(f"  Max capital: ₹{summary['Max Capital (₹Cr)']:.2f} Cr")
        print(f"  Peak positions: {summary['Peak Positions']}")

    capital_summary_df = pd.DataFrame(capital_summary)
    print("\n✓ Capital tracking metrics calculated")

    # Year-wise capital tracking
    print("\nCalculating year-wise capital metrics...")
    yearwise_capital = []

    for label in LABELS:
        if label not in capital_timelines:
            continue

        timeline = capital_timelines[label]
        timeline['year'] = pd.to_datetime(timeline['date']).dt.year

        for year in timeline['year'].unique():
            year_data = timeline[timeline['year'] == year]

            yearwise_capital.append({
                'Label': label,
                'Year': year,
                'Max Capital (₹Cr)': year_data['capital_deployed'].max() / 10000000,
                'Avg Capital (₹Cr)': year_data['capital_deployed'].mean() / 10000000,
                'Peak Positions': int(year_data['open_positions'].max()),
                'Trades Entered': len(year_data[year_data['flow_type'] == 'Investment']),
                'Trades Exited': len(year_data[year_data['flow_type'].str.contains('Exit')]),
            })

    yearwise_capital_df = pd.DataFrame(yearwise_capital)
    print("✓ Year-wise capital metrics calculated")

    # Test period capital tracking
    print("\nCalculating test period capital metrics...")
    period_capital = []

    for label in LABELS:
        if label not in all_trades:
            continue

        for period in ['Test1', 'Test2']:
            # Get trades and cashflows for this period
            period_trades = all_trades[label][all_trades[label]['test_period'] == period]

            if len(period_trades) == 0:
                continue

            # Get cashflows for these trades
            trade_ids = period_trades['trade_id'].unique()
            period_cf = all_cashflows[label][all_cashflows[label]['trade_id'].isin(trade_ids)]

            # Calculate metrics
            period_timeline = calculate_capital_metrics(period_cf, period_trades)

            period_capital.append({
                'Label': label,
                'Period': period,
                'Max Capital (₹Cr)': period_timeline['capital_deployed'].max() / 10000000,
                'Avg Capital (₹Cr)': period_timeline['capital_deployed'].mean() / 10000000,
                'Peak Positions': int(period_timeline['open_positions'].max()),
                'Total Invested (₹Cr)': period_timeline['cumulative_invested'].max() / 10000000,
                'Final P/L (₹Cr)': period_timeline['cumulative_pl'].iloc[-1] / 10000000 if len(period_timeline) > 0 else 0,
                'IRR %': all_results[label].get(f'{period}_cumulative_irr', 0),
            })

    period_capital_df = pd.DataFrame(period_capital)
    print("✓ Test period capital metrics calculated")

    print("\n" + "="*80)
    print("ALL TABLES AND METRICS READY")
    print("="*80)
    print()


    # ===== [cell 28] =====
    # ============================================================================
    # ACTUAL × PREDICTED MATRICES: trade COUNT and IRR  (across all years + both tests)
    # Rows = actual_label, Columns = predicted_label
    # ============================================================================
    print("="*80)
    print("BUILDING ACTUAL × PREDICTED COUNT & IRR MATRICES")
    print("="*80)

    ORDER = ['Ignore', 'Low', 'Medium', 'High']   # consistent display order

    # Combine all simulated trades (every prediction bucket was simulated)
    combined_trades = pd.concat(
        [all_trades[l] for l in all_trades], ignore_index=True
    ) if all_trades else pd.DataFrame()

    # Initialise matrices
    count_matrix = pd.DataFrame(0, index=ORDER, columns=ORDER, dtype=int)
    irr_matrix   = pd.DataFrame(np.nan, index=ORDER, columns=ORDER, dtype=float)

    if len(combined_trades) > 0:
        for actual in ORDER:
            for pred in ORDER:
                cell = combined_trades[
                    (combined_trades['actual_label'] == actual) &
                    (combined_trades['predicted_label'] == pred)
                ]
                count_matrix.loc[actual, pred] = len(cell)
                if len(cell) >= 2:
                    # XIRR across this cell's entry/exit cashflows (all years, both periods)
                    cf = []
                    for _, t in cell.iterrows():
                        cf.append((t['entry_date'], -t['investment']))
                        cf.append((t['exit_date'],  t['proceeds']))
                    irr_matrix.loc[actual, pred] = xirr(cf)
                elif len(cell) == 1:
                    # single trade — report its annualized return as the IRR proxy
                    irr_matrix.loc[actual, pred] = cell.iloc[0]['annualized_return']

    # Label rows/cols clearly for the report
    count_display = count_matrix.copy()
    count_display.index   = ['Actual_' + a for a in ORDER]
    count_display.columns = ['Pred_' + p for p in ORDER]
    count_display = count_display.reset_index().rename(columns={'index': 'Actual \\ Predicted'})

    irr_display = irr_matrix.round(2).copy()
    irr_display.index   = ['Actual_' + a for a in ORDER]
    irr_display.columns = ['Pred_' + p for p in ORDER]
    irr_display = irr_display.reset_index().rename(columns={'index': 'Actual \\ Predicted'})

    print("\nCOUNT matrix (rows=actual, cols=predicted):")
    print(count_display.to_string(index=False))
    print("\nIRR % matrix (rows=actual, cols=predicted):")
    print(irr_display.to_string(index=False))
    print()
    print("Diagonal = correct predictions. The Pred_High column shows what actually")
    print("happened to everything the model called High — the key conviction check.")
    print()

    # ===== [cell 29] =====
    print("="*80)
    print("GENERATING COMPREHENSIVE EXCEL REPORT WITH CAPITAL TRACKING")
    print("="*80)

    excel_file = os.path.join(OUT_DIR, 'backtest_comprehensive_report.xlsx')

    print(f"\nCreating: {excel_file}")
    print("This will take 15-20 minutes...")
    print()
    print("📊 CREATING 17 SHEETS:")
    print("  Sheets 1-11: Original analysis (KEPT)")
    print("  Sheets 12-17: NEW capital tracking analysis")
    print()

    with pd.ExcelWriter(excel_file, engine='xlsxwriter') as writer:
        workbook = writer.book

        # Define formats
        title_format = workbook.add_format({
            'bold': True,
            'font_size': 14,
            'font_color': '#4472C4',
            'bg_color': '#F2F2F2'
        })

        header_format = workbook.add_format({
            'bold': True,
            'bg_color': '#4472C4',
            'font_color': 'white',
            'border': 1
        })

        money_format = workbook.add_format({'num_format': '₹#,##0'})
        percent_format = workbook.add_format({'num_format': '0.00%'})
        number_format = workbook.add_format({'num_format': '#,##0'})

        # ========================================================================
        # SHEETS 1-11: EXISTING SHEETS (UNCHANGED)
        # ========================================================================

        # Sheet 1: Executive Summary
        print("  Creating Sheet 1: Executive Summary...")
        summary_df.to_excel(writer, sheet_name='1_Summary', index=False, startrow=1)
        worksheet = writer.sheets['1_Summary']
        worksheet.write('A1', 'BACKTEST PERFORMANCE SUMMARY', title_format)
        worksheet.set_column('A:A', 12)
        worksheet.set_column('B:H', 15)

        # Sheet 2: Year-wise Returns
        print("  Creating Sheet 2: Year-wise Returns...")
        yearwise_df.to_excel(writer, sheet_name='2_Yearwise_Returns', index=False, startrow=1)
        worksheet = writer.sheets['2_Yearwise_Returns']
        worksheet.write('A1', 'YEAR-WISE IRR RETURNS', title_format)
        worksheet.set_column('A:F', 15)

        # Sheet 3: Cumulative Returns
        print("  Creating Sheet 3: Cumulative Returns...")
        cumulative_df.to_excel(writer, sheet_name='3_Cumulative_Returns', index=False, startrow=1)
        worksheet = writer.sheets['3_Cumulative_Returns']
        worksheet.write('A1', 'TEST PERIOD CUMULATIVE RETURNS', title_format)
        worksheet.set_column('A:F', 15)

        # Sheet 3b: Actual × Predicted Count & IRR Matrices
        print("  Creating Sheet 3b: Actual x Predicted Count & IRR Matrices...")
        ws_m = workbook.add_worksheet('3b_Actual_vs_Predicted')
        writer.sheets['3b_Actual_vs_Predicted'] = ws_m

        ws_m.write('A1', 'ACTUAL (rows) x PREDICTED (cols) — TRADE COUNT', title_format)
        # write count matrix starting row 2
        count_display.to_excel(writer, sheet_name='3b_Actual_vs_Predicted',
                               index=False, startrow=2)
        n_count_rows = len(count_display)

        # IRR matrix below, with a gap
        irr_start = 2 + n_count_rows + 3
        ws_m.write(irr_start - 1, 0, 'ACTUAL (rows) x PREDICTED (cols) — IRR %', title_format)
        irr_display.to_excel(writer, sheet_name='3b_Actual_vs_Predicted',
                             index=False, startrow=irr_start)

        ws_m.set_column('A:E', 18)
        note_row = irr_start + len(irr_display) + 3
        ws_m.write(note_row, 0,
                   'Rows = actual conviction label, Columns = predicted label.')
        ws_m.write(note_row + 1, 0,
                   'Diagonal = correct predictions. IRR computed via XIRR across all years & both test periods.')
        ws_m.write(note_row + 2, 0,
                   'Cells with <2 trades show the single trade annualized return; blank = no trades.')

        # Sheet 4: Detailed Metrics
        print("  Creating Sheet 4: Detailed Metrics...")
        metrics_df.to_excel(writer, sheet_name='4_Detailed_Metrics', index=False, startrow=1)
        worksheet = writer.sheets['4_Detailed_Metrics']
        worksheet.write('A1', 'DETAILED PERFORMANCE METRICS', title_format)
        worksheet.set_column('A:P', 16)

        # Sheet 5: ALL Trades Combined
        print("  Creating Sheet 5: All Trades (this may take a while)...")
        all_trades_combined = pd.concat([df for df in all_trades.values()], ignore_index=True)
        all_trades_combined = all_trades_combined.sort_values('entry_date')
        all_trades_combined.to_excel(writer, sheet_name='5_All_Trades', index=False, startrow=1)
        worksheet = writer.sheets['5_All_Trades']
        worksheet.write('A1', f'ALL TRADES LOG ({len(all_trades_combined):,} trades)', title_format)
        worksheet.set_column('A:Z', 14)
        worksheet.autofilter(1, 0, len(all_trades_combined)+1, len(all_trades_combined.columns)-1)

        # Sheet 6: ALL Cashflows
        print("  Creating Sheet 6: All Cashflows...")
        all_cashflows_combined = pd.concat([df for df in all_cashflows.values()], ignore_index=True)
        all_cashflows_combined = all_cashflows_combined.sort_values('date')
        all_cashflows_combined.to_excel(writer, sheet_name='6_All_Cashflows', index=False, startrow=1)
        worksheet = writer.sheets['6_All_Cashflows']
        worksheet.write('A1', f'ALL CASHFLOWS ({len(all_cashflows_combined):,} entries)', title_format)
        worksheet.set_column('A:H', 16)
        worksheet.autofilter(1, 0, len(all_cashflows_combined)+1, len(all_cashflows_combined.columns)-1)

        # Sheet 7: Stock Performance
        print("  Creating Sheet 7: Stock Performance...")
        stock_performance_df.to_excel(writer, sheet_name='7_Stock_Performance', index=False, startrow=1)
        worksheet = writer.sheets['7_Stock_Performance']
        worksheet.write('A1', 'STOCK-LEVEL PERFORMANCE', title_format)
        worksheet.set_column('A:K', 16)
        worksheet.autofilter(1, 0, len(stock_performance_df)+1, len(stock_performance_df.columns)-1)

        # Sheets 8-11: Individual Label Details
        sheet_num = 8
        for label in LABELS:
            if label not in all_trades:
                continue

            print(f"  Creating Sheet {sheet_num}: {label} Details...")
            sheet_name = f'{sheet_num}_{label}_Details'

            trades = all_trades[label]
            trades.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
            worksheet = writer.sheets[sheet_name]
            worksheet.write('A1', f'{label.upper()} CONVICTION TRADES ({len(trades):,})', title_format)
            worksheet.set_column('A:Z', 14)
            worksheet.autofilter(1, 0, len(trades)+1, len(trades.columns)-1)
            sheet_num += 1

        # ========================================================================
        # SHEETS 12-17: NEW CAPITAL TRACKING SHEETS
        # ========================================================================

        print("\n  📊 NEW CAPITAL TRACKING SHEETS:")

        # Sheet 12: Overall Capital Summary
        print("  Creating Sheet 12: Overall Capital Summary...")
        capital_summary_df.to_excel(writer, sheet_name='12_Capital_Summary', index=False, startrow=1)
        worksheet = writer.sheets['12_Capital_Summary']
        worksheet.write('A1', 'OVERALL CAPITAL DEPLOYMENT SUMMARY', title_format)
        worksheet.set_column('A:F', 18)

        # Add conditional formatting for max capital
        max_row = len(capital_summary_df) + 2
        worksheet.conditional_format(f'B2:B{max_row}', {
            'type': '3_color_scale',
            'min_color': '#63BE7B',
            'mid_color': '#FFEB84',
            'max_color': '#F8696B'
        })

        # Sheet 13: Year-wise Capital Tracking
        print("  Creating Sheet 13: Year-wise Capital Tracking...")
        yearwise_capital_df.to_excel(writer, sheet_name='13_Yearwise_Capital', index=False, startrow=1)
        worksheet = writer.sheets['13_Yearwise_Capital']
        worksheet.write('A1', 'YEAR-WISE CAPITAL DEPLOYMENT', title_format)
        worksheet.set_column('A:G', 16)
        worksheet.autofilter(1, 0, len(yearwise_capital_df)+1, len(yearwise_capital_df.columns)-1)

        # Sheet 14: Test Period Capital Summary
        print("  Creating Sheet 14: Test Period Capital Summary...")
        period_capital_df.to_excel(writer, sheet_name='14_Period_Capital', index=False, startrow=1)
        worksheet = writer.sheets['14_Period_Capital']
        worksheet.write('A1', 'TEST PERIOD CAPITAL DEPLOYMENT', title_format)
        worksheet.set_column('A:H', 18)

        # Sheet 15-17: Detailed Cashflow Timelines (one per key label)
        sheet_num = 15
        for label in ['High', 'Medium', 'Low']:  # Only top 3 to avoid too many sheets
            if label not in capital_timelines:
                continue

            print(f"  Creating Sheet {sheet_num}: {label} Cashflow Timeline...")
            sheet_name = f'{sheet_num}_{label}_Timeline'

            timeline = capital_timelines[label].copy()
            # Select key columns for display
            timeline_display = timeline[[
                'date', 'symbol', 'flow_type', 'amount',
                'open_positions', 'capital_deployed', 'cumulative_pl'
            ]].copy()

            timeline_display.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
            worksheet = writer.sheets[sheet_name]
            worksheet.write('A1', f'{label.upper()} - DETAILED CASHFLOW TIMELINE', title_format)
            worksheet.set_column('A:G', 16)
            worksheet.autofilter(1, 0, len(timeline_display)+1, len(timeline_display.columns)-1)

            # Add chart for capital deployment over time
            if len(timeline_display) > 10:  # Only add chart if enough data
                chart = workbook.add_chart({'type': 'line'})
                chart.add_series({
                    'name': 'Capital Deployed',
                    'categories': [sheet_name, 2, 0, len(timeline_display)+1, 0],
                    'values': [sheet_name, 2, 5, len(timeline_display)+1, 5],
                    'line': {'color': '#4472C4', 'width': 2}
                })
                chart.set_title({'name': f'{label} - Capital Deployed Over Time'})
                chart.set_x_axis({'name': 'Date'})
                chart.set_y_axis({'name': 'Capital (₹)', 'num_format': '₹#,##0'})
                chart.set_size({'width': 720, 'height': 400})
                worksheet.insert_chart('I2', chart)

            sheet_num += 1

    print("\n✓ Comprehensive Excel report created")
    print(f"  File: {excel_file}")
    print(f"  Size: {os.path.getsize(excel_file)/(1024*1024):.2f} MB")
    print(f"\n📊 TOTAL SHEETS: {sheet_num - 1}")
    print("  • Sheets 1-11: Original analysis")
    print("  • Sheets 12-14: Capital summary tables")
    print(f"  • Sheets 15-{sheet_num-1}: Detailed timelines with charts")
    print()

    # ===== comparison rows: REUSE the mark-to-market IRRs computed above =====
    # all_results[cls] holds, per predicted class:
    #   'year_irrs'            → {year: mark-to-market IRR}   (matches Excel yearly tab)
    #   'TestN_cumulative_irr' → mark-to-market IRR for the test period
    # We tabulate these EXACT values so the comparison workbook and the per-model
    # Excel agree by construction (same numbers, not a re-computation).
    comp_rows = []
    PRED_CLASSES = ['Ignore', 'Low', 'Medium', 'High']
    YEARS = [2020, 2021, 2022, 2023, 2024, 2025]

    # trade counts per (class, segment) from combined_trades (counts only; IRRs come
    # from all_results). Counting is unaffected by mark-to-market.
    _ct = combined_trades.copy() if len(combined_trades) else pd.DataFrame()
    if len(_ct):
        _ct['entry_year'] = pd.to_datetime(_ct['entry_date']).dt.year

    def _count(cls, seg):
        if not len(_ct): return 0
        d = _ct[_ct['predicted_label'] == cls]
        if seg == 'Aggregate':      return len(d)
        if seg.startswith('Year_'): return len(d[d['entry_year'] == int(seg.split('_')[1])])
        return len(d[d['test_period'] == seg])   # Test1 / Test2

    # Build one row per (class-segment) is not our layout; our layout is one row per
    # segment with the 4 class IRRs as columns. Assemble that:
    SEGMENTS = ['Aggregate', 'Test1', 'Test2'] + [f'Year_{y}' for y in YEARS]
    for seg in SEGMENTS:
        row = {'model': MODEL_LABEL, 'segment': seg}
        total = 0
        for cls in PRED_CLASSES:
            res = all_results.get(cls, {})
            if seg == 'Aggregate':
                # Aggregate IRR isn't a simple combine of mark-to-market pieces;
                # report it as blank to avoid a misleading number. Counts still shown.
                irr = None
            elif seg.startswith('Year_'):
                yr = int(seg.split('_')[1])
                irr = res.get('year_irrs', {}).get(yr, None)
            else:  # Test1 / Test2
                irr = res.get(f'{seg}_cumulative_irr', None)
            row[f'{cls}_IRR']   = round(irr, 2) if irr is not None else None
            row[f'{cls}_count'] = _count(cls, seg)
            total += row[f'{cls}_count']
        row['total_trades'] = total
        comp_rows.append(row)

    print(f"\n✓ {MODEL_LABEL} complete → {OUT_DIR}")
    return comp_rows


## **STEP 3d: Run all models + compile comparison workbook**

In [7]:
print("="*80); print("BATCH BACKTEST — ALL COMBO MODELS"); print("="*80)
import time as _time
_t0=_time.time()

all_comparison_rows=[]
for k, mpath in enumerate(combo_model_files, 1):
    label = os.path.splitext(os.path.basename(mpath))[0]   # e.g. model_combo_001
    out_dir = os.path.join(BATCH_OUTPUT_DIR, label)
    print(f"\n[{k}/{len(combo_model_files)}] {label}  (elapsed {(_time.time()-_t0)/60:.1f} min)")
    try:
        rows = run_one_backtest(mpath, out_dir, label)
        all_comparison_rows.extend(rows)
    except Exception as e:
        print(f"  ✗ {label} FAILED: {e}")
        import traceback; traceback.print_exc()

print(f"\n{'='*80}\nALL MODELS DONE in {(_time.time()-_t0)/60:.1f} min\n{'='*80}")

# ── Comparison workbook: one sheet (model×segment rows), + pivoted sheets per class ──
if not all_comparison_rows:
    raise RuntimeError(
        "No models completed successfully — comparison workbook cannot be built. "
        "Check the per-model error messages above (commonly a missing shared file "
        "like categorical_encoders.pkl). Fix the cause and re-run.")

comp_df = pd.DataFrame(all_comparison_rows)
# tidy column order (only keep columns that exist, to be robust)
PRED=['Ignore','Low','Medium','High']
front=['model','segment','total_trades']
irr_cols=[f'{c}_IRR' for c in PRED]
cnt_cols=[f'{c}_count' for c in PRED]
wanted = front+irr_cols+cnt_cols
comp_df = comp_df[[c for c in wanted if c in comp_df.columns]]
print(f"Comparison rows: {len(comp_df)} from "
      f"{comp_df['model'].nunique() if 'model' in comp_df else 0} model(s)")

comp_path=os.path.join(BATCH_OUTPUT_DIR, 'cross_model_comparison.xlsx')
with pd.ExcelWriter(comp_path, engine='xlsxwriter') as xw:
    # Sheet 1: long form, model × segment
    comp_df.to_excel(xw, sheet_name='By_Model_Segment', index=False)
    # Pivoted: one sheet per predicted class, rows=model, cols=segment (IRR)
    SEG_ORDER=['Aggregate','Test1','Test2','Year_2020','Year_2021','Year_2022',
               'Year_2023','Year_2024','Year_2025']
    for cls in PRED:
        piv=comp_df.pivot_table(index='model', columns='segment',
                                values=f'{cls}_IRR', aggfunc='first')
        piv=piv.reindex(columns=[s for s in SEG_ORDER if s in piv.columns])
        piv.to_excel(xw, sheet_name=f'{cls}_IRR_pivot')
print(f"✓ cross_model_comparison.xlsx ({len(comp_df)} rows)")
print(f"  Sheet 'By_Model_Segment' + per-class IRR pivot sheets")

BATCH BACKTEST — ALL COMBO MODELS

[1/6] model_combo_062  (elapsed 0.0 min)

################################################################################
# BACKTEST: model_combo_062
################################################################################
--------------------------------------------------------------------------------
LOADING MODEL
--------------------------------------------------------------------------------
✓ Model: XGBClassifier
✓ Encoders: 0 features
✓ label_mapping.pkl loaded from Code 8b
  Mapping: {'Ignore': 0, 'Low': 1, 'Medium': 2, 'High': 3}

--------------------------------------------------------------------------------
LOADING TEST DATA
--------------------------------------------------------------------------------
Features: 275

Loading Test1 (2020-2022)...
  ✓ 112,048 rows

Loading Test2 (2023-2025)...
  ✓ 110,999 rows

Total: 223,047 rows
  ✓ Using 'ATR_14' as ATR for exit calculation

Date range: 2020-01-01 00:00:00 to 2025-12-31 00:00:00

## **STEP 3e: Zip all outputs + auto-download**

In [8]:
print("="*80); print("ZIPPING BATCH OUTPUTS"); print("="*80)
from google.colab import files
import shutil

# Each model's full output lives in batch_outputs/model_combo_XXX/.
# Show the tree for transparency.
for root,_,fs in os.walk(BATCH_OUTPUT_DIR):
    rel=os.path.relpath(root, BATCH_OUTPUT_DIR)
    if fs:
        print(f"  {rel}/")
        for fn in sorted(fs):
            print(f"     {fn}")

zip_path=shutil.make_archive('/content/batch_backtest_outputs','zip',BATCH_OUTPUT_DIR)
print(f"\n✓ Zipped ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)")
files.download(zip_path)
print("✓ Downloading batch_backtest_outputs.zip")
print("\nContents: one folder per model (full Excel + 9 CSVs each) +")
print("          cross_model_comparison.xlsx at the root.")

ZIPPING BATCH OUTPUTS
  ./
     cross_model_comparison.xlsx
  model_combo_080/
     backtest_comprehensive_report.xlsx
     stock_performance.csv
     trades_High.csv
     trades_Ignore.csv
     trades_Low.csv
     trades_Medium.csv
  model_combo_074/
     backtest_comprehensive_report.xlsx
     stock_performance.csv
     trades_High.csv
     trades_Ignore.csv
     trades_Low.csv
     trades_Medium.csv
  model_combo_077/
     backtest_comprehensive_report.xlsx
     stock_performance.csv
     trades_High.csv
     trades_Ignore.csv
     trades_Low.csv
     trades_Medium.csv
  model_combo_071/
     backtest_comprehensive_report.xlsx
     stock_performance.csv
     trades_High.csv
     trades_Ignore.csv
     trades_Low.csv
     trades_Medium.csv
  model_combo_068/
     backtest_comprehensive_report.xlsx
     stock_performance.csv
     trades_High.csv
     trades_Ignore.csv
     trades_Low.csv
     trades_Medium.csv
  model_combo_062/
     backtest_comprehensive_report.xlsx
     stock_perfo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloading batch_backtest_outputs.zip

Contents: one folder per model (full Excel + 9 CSVs each) +
          cross_model_comparison.xlsx at the root.
